# Recommandation d'artistes avec XGBoost

Toute la logique se trouve dans `src/recommend_xgboost.py`. Ce notebook charge une version précise des données, sélectionne la meilleure configuration sur la validation, entraîne le modèle final et montre son utilisation.

In [1]:
import os
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "8")

import importlib
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / "src").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Impossible de trouver la racine MusicRecoSystem")
sys.path.insert(0, str(PROJECT_ROOT))

import src.recommend_xgboost as xgb_module
xgb_module = importlib.reload(xgb_module)

## 1. Charger les données nettoyées

In [2]:
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "listens.parquet"
listens = pd.read_parquet(DATA_PATH)

pd.Series({
    "dataset_version": listens["dataset_version"].iloc[0],
    "interactions": len(listens),
    "users": listens["user"].nunique(),
    "artist_identities": listens["artist_uid"].nunique(),
})

dataset_version      clean-v1.0.0-19b966482277
interactions                             10064
users                                       82
artist_identities                         2640
dtype: object

## 2. Sélectionner la configuration et entraîner le modèle final

Les configurations sont comparées avec le NDCG@10 de validation. Le meilleur modèle est ensuite réentraîné sur train + validation et évalué sur le test.

In [3]:
import src.recommend_xgboost as xgb_module
xgb_module = importlib.reload(xgb_module)

experiment = xgb_module.select_and_fit_best_xgboost(
    listens,
    top_n=10,
    random_state=42,
    verbose=True,
)

display(experiment.tuning_results)
display(pd.DataFrame({
    "validation": experiment.validation_metrics,
    "test_final": experiment.test_metrics,
}))
print(f"Dataset : {experiment.dataset_version}")
print(f"Meilleure configuration : {experiment.best_config}")

XGBoost 1/3 — balanced
  Recall@10=0.1575 | NDCG@10=0.1084
XGBoost 2/3 — shallow
  Recall@10=0.1649 | NDCG@10=0.1111
XGBoost 3/3 — regularized
  Recall@10=0.1538 | NDCG@10=0.1085
Meilleure configuration : shallow


,config,n_estimators,max_depth,learning_rate,recall@10,ndcg@10,n_evaluable_users
0,balanced,100,3,0.05,0.157487,0.108430,67
1,shallow,150,2,0.05,0.164950,0.111143,67
2,regularized,200,3,0.03,0.153756,0.108507,67


,validation,test_final
recall@10,0.164950,0.116282
ndcg@10,0.111143,0.092489
n_evaluable_users,67.000000,66.000000


Dataset : clean-v1.0.0-19b966482277
Meilleure configuration : shallow


## 3. Comprendre les variables du modèle

In [4]:
experiment.recommender.feature_importance().to_frame()

,importance
neighbor_affinity_k20,0.654318
artist_popularity,0.238249
neighbor_affinity_k5,0.053860
user_diversity,0.032974
user_activity,0.020598


## 4. Obtenir 10 recommandations

In [5]:
USER = "BKILPATRICK11"

experiment.recommender.recommend(USER, top_n=10)

,rank,artist_uid,artist,score
0,1,mbid:e520459c-dff4-491d-a6e4-c97be35e0044,Frank Ocean,0.971191
1,2,mbid:9fff2f8a-21e6-47de-a2b8-7f449929d43f,Drake,0.970190
2,3,mbid:8837b875-3689-4646-b3fd-d8b53815c7a8,Lil Uzi Vert,0.871639
3,4,mbid:c8b03190-306c-4120-bb0b-6f2ebfc06ea9,The Weeknd,0.857357
4,5,mbid:201412c6-7ced-472b-87a9-9259ec3c07d3,Brent Faiyaz,0.847133
5,6,mbid:21645c31-fe1c-45a4-955c-3e172b12c3f9,Freddie Gibbs,0.847133
6,7,mbid:272989c8-5535-492d-a25c-9f58803e027f,SZA,0.847133
7,8,mbid:4e024037-14b7-4aea-99ad-c6ace63b9620,Madvillain,0.847133
8,9,mbid:7441014f-f8f5-494f-81db-ff166fbc078d,PinkPantheress,0.847133
9,10,mbid:b46e298d-d3a5-4695-a03e-63e8c523b23e,Daniel Caesar,0.847133
